In [1]:
from google.colab import drive
import requests, tqdm
import pandas as pd
from os.path import join
import pandas as pd
import re

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:


# Your paths
textfile_path = '/content/drive/MyDrive/marl'
links_file_path = f"{textfile_path}/sa1b_links.txt"

# Read the links
df = pd.read_csv(links_file_path, sep="\t", header=None, names=["file", "url"])

print(f"Total files in links: {len(df)}")
print("\nFirst few files:")
print(df.head(10))

# Filter for ONLY Stage 1+2 (sa_000000 to sa_000299)
def is_stage1_2(filename):
    """Check if file is from Stage 1 or 2 (first 300 folders)."""
    match = re.search(r'sa_(\d{6})', filename)
    if match:
        folder_num = int(match.group(1))
        return folder_num < 300  # Stage 1+2 are folders 0-299
    return False

# Filter the dataframe
df_stage1_2 = df[df['file'].apply(is_stage1_2)]

print(f"\nStage 1+2 files: {len(df_stage1_2)}")
print(f"Stage 3+ files (excluded): {len(df) - len(df_stage1_2)}")

# Save filtered links
stage1_2_links_path = f"{textfile_path}/sa1b_stage1_2_links.txt"
df_stage1_2["url"].to_csv(stage1_2_links_path, index=False, header=False)

print(f"\nFiltered links saved to: {stage1_2_links_path}")
print("\nFirst 5 Stage 1+2 URLs:")
print(df_stage1_2.head())

Total files in links: 1002

First few files:
            file                                                url
0  sa_000020.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An_YmP5O...
1  sa_000021.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An-V_ojE...
2  sa_000022.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An-MkPJ4...
3  sa_000023.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An-vK2KX...
4  sa_000024.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An_IKmGm...
5  sa_000025.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An8tb2zR...
6  sa_000026.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An9lYPUP...
7  sa_000027.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An92v0YH...
8  sa_000028.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An89aWnx...
9  sa_000029.tar  https://scontent.xx.fbcdn.net/m1/v/t6/An_HvqW2...

Stage 1+2 files: 300
Stage 3+ files (excluded): 702

Filtered links saved to: /content/drive/MyDrive/marl/sa1b_stage1_2_links.txt

First 5 Stage 1+2 URLs:
            file                                   

In [5]:
# Direct download
ids_url = "https://scontent.xx.fbcdn.net/m1/v/t6/An-WXpFAE-ykzczGr6_3ZVZgDS4t9TOwSpFgre5g_6l5uXdzu2f9P5l7JHie5DpZE-cm_Pgm5Yz5A1S92iZQIjuAg3AcVRGkPg.txt?_nc_gid&ccb=10-5&oh=00_AfiyRNI2mnU6FBbG-WFB0LyjE3orZLgphEJZj5o_3qOcpA&oe=693A4322&_nc_sid=0fdd51"

output_path = '/content/drive/MyDrive/sam_dataset/sa_images_ids.txt'

!wget -O "$output_path" "$ids_url"

print("✓ Downloaded!")

--2025-11-26 18:37:56--  https://scontent.xx.fbcdn.net/m1/v/t6/An-WXpFAE-ykzczGr6_3ZVZgDS4t9TOwSpFgre5g_6l5uXdzu2f9P5l7JHie5DpZE-cm_Pgm5Yz5A1S92iZQIjuAg3AcVRGkPg.txt?_nc_gid&ccb=10-5&oh=00_AfiyRNI2mnU6FBbG-WFB0LyjE3orZLgphEJZj5o_3qOcpA&oe=693A4322&_nc_sid=0fdd51
Resolving scontent.xx.fbcdn.net (scontent.xx.fbcdn.net)... 157.240.254.7, 2a03:2880:f357:80:face:b00c:0:3
Connecting to scontent.xx.fbcdn.net (scontent.xx.fbcdn.net)|157.240.254.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 89559366 (85M)
Saving to: ‘/content/drive/MyDrive/sam_dataset/sa_images_ids.txt’

/content/drive/MyDr 100%[===================>]  85.41M  35.2MB/s    in 2.4s    

2025-11-26 18:38:00 (35.2 MB/s) - ‘/content/drive/MyDrive/sam_dataset/sa_images_ids.txt’ saved [89559366/89559366]

✓ Downloaded!


In [4]:
import os
from tqdm import tqdm

def download_stage1_2_with_validation(links_path, output_dir):
    """Download Stage 1+2 with progress tracking and validation."""

    # Read links
    with open(links_path, 'r') as f:
        urls = [line.strip() for line in f if line.strip()]

    print(f"Total files to download: {len(urls)}")

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Download each file
    successful = 0
    failed = []

    for url in tqdm(urls, desc="Downloading Stage 1+2"):
        filename = url.split('/')[-1]
        output_path = os.path.join(output_dir, filename)

        # Skip if already downloaded
        if os.path.exists(output_path):
            file_size = os.path.getsize(output_path)
            if file_size > 1024:  # At least 1KB
                print(f"✓ Already downloaded: {filename}")
                successful += 1
                continue

        # Download
        try:
            !wget -q -c "$url" -O "$output_path"

            # Verify download
            if os.path.exists(output_path) and os.path.getsize(output_path) > 1024:
                successful += 1
            else:
                failed.append(filename)
                print(f"✗ Failed: {filename}")
        except Exception as e:
            failed.append(filename)
            print(f"✗ Error downloading {filename}: {e}")

    print(f"\n{'='*50}")
    print(f"Download Summary:")
    print(f"  Successful: {successful}/{len(urls)}")
    print(f"  Failed: {len(failed)}")
    if failed:
        print(f"  Failed files: {failed[:10]}...")  # Show first 10
    print(f"{'='*50}")

    return successful, failed

# Run download
output_dir = '/content/drive/MyDrive/sam_dataset/stage1_2'
stage1_2_links_path = f"{textfile_path}/sub_sa1b_stage1_2_links.txt"
successful, failed = download_stage1_2_with_validation(
    stage1_2_links_path,
    output_dir
)

Total files to download: 5


^C


^C


^C


^C


^C
✗ Failed: An_IKmGmpEiuWd7Ja3MKBqYwtVAcPcvpH_iGBVzSZahbhAwZxAWmFFdwbPo1MFZh3pIaVabv3ucWH0NVhpxAXBkPpkSA66s.tar?_nc_gid&ccb=10-5&oh=00_AfgpEK9MddL6BLknlizukuYvxnTCfglnERUHb1kYpMeZ8g&oe=693A53B3&_nc_sid=0fdd51

Download Summary:
  Successful: 4/5
  Failed: 1
  Failed files: ['An_IKmGmpEiuWd7Ja3MKBqYwtVAcPcvpH_iGBVzSZahbhAwZxAWmFFdwbPo1MFZh3pIaVabv3ucWH0NVhpxAXBkPpkSA66s.tar?_nc_gid&ccb=10-5&oh=00_AfgpEK9MddL6BLknlizukuYvxnTCfglnERUHb1kYpMeZ8g&oe=693A53B3&_nc_sid=0fdd51']...


In [4]:
import tarfile
import os
from pathlib import Path
from tqdm import tqdm

def extract_sa1b_archives(archive_dir, extract_dir):
    """Extract all .tar files from Stage 1+2."""

    archive_dir = Path(archive_dir)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # Find all tar files
    tar_files = sorted(archive_dir.glob("sa_*.tar"))
    print(f"Found {len(tar_files)} tar archives")

    for tar_path in tqdm(tar_files, desc="Extracting"):
        folder_name = tar_path.stem  # e.g., "sa_000001"
        output_folder = extract_dir / folder_name

        # Skip if already extracted
        if output_folder.exists() and len(list(output_folder.iterdir())) > 0:
            print(f"✓ Already extracted: {folder_name}")
            continue

        try:
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(extract_dir)
            print(f"✓ Extracted: {folder_name}")

            # Optional: remove tar file after extraction to save space
            # tar_path.unlink()

        except Exception as e:
            print(f"✗ Failed to extract {folder_name}: {e}")

# Extract
archive_dir = '/content/drive/MyDrive/sam_dataset/stage1_2'
extract_dir = '/content/drive/MyDrive/sam_dataset/extracted'

extract_sa1b_archives(archive_dir, extract_dir)

Found 5 tar archives


Extracting:   0%|          | 0/5 [00:00<?, ?it/s]/tmp/ipython-input-1257187091.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)
Extracting:  20%|██        | 1/5 [09:16<37:06, 556.59s/it]

✓ Extracted: sa_000020


Extracting:  40%|████      | 2/5 [18:54<28:27, 569.06s/it]

✓ Extracted: sa_000021


...

✓ Extracted: sa_000022


Extracting:  80%|████████  | 4/5 [37:48<09:27, 567.63s/it]

✓ Extracted: sa_000023


Extracting: 100%|██████████| 5/5 [47:21<00:00, 568.35s/it]

✓ Extracted: sa_000024


In [6]:
def verify_sa1b_structure(extract_dir):
    """Verify SA-1B Stage 1+2 structure."""

    extract_dir = Path(extract_dir)

    # Check folders sa_000000 to sa_000299
    folders_found = []
    folders_missing = []

    for i in range(300):
        folder_name = f"sa_{i:06d}"
        folder_path = extract_dir / folder_name

        if folder_path.exists():
            # Count images and JSONs
            images = list(folder_path.glob("*.jpg"))
            jsons = list(folder_path.glob("*.json"))

            folders_found.append({
                'folder': folder_name,
                'images': len(images),
                'jsons': len(jsons)
            })
        else:
            folders_missing.append(folder_name)

    print(f"{'='*50}")
    print(f"SA-1B Stage 1+2 Verification:")
    print(f"  Folders found: {len(folders_found)}/300")
    print(f"  Folders missing: {len(folders_missing)}")
    print(f"{'='*50}")

    if folders_found:
        total_images = sum(f['images'] for f in folders_found)
        total_jsons = sum(f['jsons'] for f in folders_found)
        print(f"  Total images: {total_images:,}")
        print(f"  Total annotations: {total_jsons:,}")
        print(f"  Average per folder: {total_images/len(folders_found):.1f} images")

    if folders_missing:
        print(f"\n  Missing folders: {folders_missing[:10]}...")

    return folders_found, folders_missing

# Verify
extract_dir = '/content/drive/MyDrive/sam_dataset/extracted'
folders_found, folders_missing = verify_sa1b_structure(extract_dir)

SA-1B Stage 1+2 Verification:
  Folders found: 0/300
  Folders missing: 300

  Missing folders: ['sa_000000', 'sa_000001', 'sa_000002', 'sa_000003', 'sa_000004', 'sa_000005', 'sa_000006', 'sa_000007', 'sa_000008', 'sa_000009']...


In [12]:
import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
from pycocotools import mask as mask_utils
from tqdm import tqdm

# Your extracted folder path
EXTRACTED_DIR = '/content/drive/MyDrive/sam_dataset/minibatch'

def decode_sa1b_mask(annotation):
    """Decode SA-1B RLE mask to binary mask."""
    segmentation = annotation['segmentation']
    if isinstance(segmentation, dict):
        mask = mask_utils.decode(segmentation)
    else:
        raise ValueError(f"Unexpected segmentation format: {type(segmentation)}")
    return mask.astype(bool)

def calculate_iou(pred_mask, gt_mask):
    """Calculate Intersection over Union."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return 0.0
    return intersection / union

def calculate_dice(pred_mask, gt_mask):
    """Calculate Dice coefficient."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    if total == 0:
        return 0.0
    return 2 * intersection / total

def calculate_metrics(pred_mask, gt_mask):
    """Calculate comprehensive metrics."""
    iou = calculate_iou(pred_mask, gt_mask)
    dice = calculate_dice(pred_mask, gt_mask)

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    pred_area = pred_mask.sum()
    gt_area = gt_mask.sum()

    precision = intersection / pred_area if pred_area > 0 else 0
    recall = intersection / gt_area if gt_area > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'iou': iou,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

def load_sa1b_image_and_annotations(json_path):
    """Load image and annotations from SA-1B format."""
    json_path = Path(json_path)

    # Load JSON
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Load image
    img_path = json_path.with_suffix('.jpg')
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")

    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    return image, data

def evaluate_sam_on_annotation(image, annotation, predictor):
    """Evaluate SAM on a single annotation using its bounding box as prompt."""

    # Get ground truth mask
    gt_mask = decode_sa1b_mask(annotation)

    # Get bounding box from annotation
    bbox = annotation['bbox']  # [x, y, width, height]
    input_box = np.array([bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]])

    # SAM prediction
    predictor.set_image(image)

    with torch.no_grad():
        masks, scores, logits = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=input_box[None, :],
            multimask_output=False,
        )

    pred_mask = masks[0]
    sam_score = scores[0]

    # Calculate metrics
    metrics = calculate_metrics(pred_mask, gt_mask)
    metrics['sam_score'] = sam_score
    metrics['gt_predicted_iou'] = annotation.get('predicted_iou', 0)
    metrics['gt_stability_score'] = annotation.get('stability_score', 0)
    metrics['gt_area'] = annotation.get('area', gt_mask.sum())

    predictor.reset_image()

    return pred_mask, gt_mask, metrics, input_box

def visualize_comparison(image, gt_mask, pred_mask, metrics, bbox, title=""):
    """Visualize ground truth vs prediction."""

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Original image
    axes[0, 0].imshow(image)
    axes[0, 0].set_title("Original Image")
    axes[0, 0].axis('off')

    # Image with bbox
    img_with_bbox = image.copy()
    x1, y1, x2, y2 = bbox.astype(int)
    cv2.rectangle(img_with_bbox, (x1, y1), (x2, y2), (255, 0, 0), 3)
    axes[0, 1].imshow(img_with_bbox)
    axes[0, 1].set_title("Bounding Box (SAM Input)")
    axes[0, 1].axis('off')

    # Ground truth
    axes[0, 2].imshow(image)
    axes[0, 2].imshow(gt_mask, alpha=0.5, cmap='Greens')
    axes[0, 2].set_title("Ground Truth (SA-1B)")
    axes[0, 2].axis('off')

    # SAM prediction
    axes[1, 0].imshow(image)
    axes[1, 0].imshow(pred_mask, alpha=0.5, cmap='Blues')
    axes[1, 0].set_title("SAM Prediction")
    axes[1, 0].axis('off')

    # Overlap
    overlap = np.zeros((*gt_mask.shape, 3))
    overlap[gt_mask, 0] = 1  # GT in red
    overlap[pred_mask, 2] = 1  # Pred in blue
    overlap[np.logical_and(gt_mask, pred_mask)] = [1, 1, 0]  # Overlap in yellow

    axes[1, 1].imshow(image)
    axes[1, 1].imshow(overlap, alpha=0.5)
    axes[1, 1].set_title(f"Overlap (IoU: {metrics['iou']:.3f})")
    axes[1, 1].axis('off')

    # Metrics text
    axes[1, 2].axis('off')
    metrics_text = f"""
    {title}

    METRICS:
    IoU: {metrics['iou']:.4f}
    Dice: {metrics['dice']:.4f}
    Precision: {metrics['precision']:.4f}
    Recall: {metrics['recall']:.4f}
    F1: {metrics['f1']:.4f}

    SAM Score: {metrics['sam_score']:.4f}

    GT Quality:
    Pred IoU: {metrics['gt_predicted_iou']:.4f}
    Stability: {metrics['gt_stability_score']:.4f}
    Area: {metrics['gt_area']:.0f} px
    """
    axes[1, 2].text(0.1, 0.5, metrics_text, fontsize=12,
                    verticalalignment='center', family='monospace')

    plt.tight_layout()
    plt.show()

def evaluate_sam_on_folder(extracted_dir, predictor, num_images=5,
                           min_gt_quality=0.90, num_masks_per_image=3):
    """
    Evaluate SAM on random images from extracted folder.

    Args:
        extracted_dir: Path to extracted SA-1B folder
        predictor: SAM predictor instance
        num_images: Number of images to evaluate
        min_gt_quality: Minimum GT quality (predicted_iou) to consider
        num_masks_per_image: Number of masks to evaluate per image
    """

    extracted_dir = Path(extracted_dir)

    # Find all JSON files
    json_files = list(extracted_dir.glob("**/*.json"))
    print(f"Found {len(json_files)} annotation files")

    if len(json_files) == 0:
        print("No JSON files found! Check your path.")
        return

    # Randomly sample images
    import random
    random.shuffle(json_files)

    all_metrics = []
    images_evaluated = 0

    for json_path in json_files:
        if images_evaluated >= num_images:
            break

        try:
            # Load image and annotations
            image, data = load_sa1b_image_and_annotations(json_path)

            # Filter high-quality annotations
            annotations = [
                ann for ann in data['annotations']
                if ann.get('predicted_iou', 0) >= min_gt_quality
                and ann.get('stability_score', 0) >= min_gt_quality
                and ann.get('area', 0) >= 500
            ]

            if len(annotations) == 0:
                continue

            print(f"\n{'='*80}")
            print(f"Image {images_evaluated + 1}/{num_images}: {json_path.name}")
            print(f"Image size: {data['image']['width']}x{data['image']['height']}")
            print(f"High-quality masks available: {len(annotations)}")
            print(f"{'='*80}")

            # Evaluate on multiple masks from this image
            num_to_eval = min(num_masks_per_image, len(annotations))
            selected_anns = random.sample(annotations, num_to_eval)

            for i, ann in enumerate(selected_anns):
                print(f"\n  Mask {i+1}/{num_to_eval}:")

                # Evaluate
                pred_mask, gt_mask, metrics, bbox = evaluate_sam_on_annotation(
                    image, ann, predictor
                )

                # Store metrics
                all_metrics.append(metrics)

                # Print metrics
                print(f"    IoU: {metrics['iou']:.4f} | Dice: {metrics['dice']:.4f} | "
                      f"F1: {metrics['f1']:.4f} | SAM Score: {metrics['sam_score']:.4f}")

                # Visualize
                visualize_comparison(
                    image, gt_mask, pred_mask, metrics, bbox,
                    title=f"{json_path.name} - Mask {i+1}/{num_to_eval}"
                )

                # Memory cleanup
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            images_evaluated += 1

        except Exception as e:
            print(f"Error processing {json_path.name}: {e}")
            continue

    # Summary statistics
    if all_metrics:
        print(f"\n{'='*80}")
        print("SUMMARY STATISTICS")
        print(f"{'='*80}")
        print(f"Total masks evaluated: {len(all_metrics)}")
        print(f"Images evaluated: {images_evaluated}")
        print(f"\nAverage Metrics:")

        for key in ['iou', 'dice', 'precision', 'recall', 'f1', 'sam_score']:
            values = [m[key] for m in all_metrics]
            print(f"  {key.upper():15s}: {np.mean(values):.4f} ± {np.std(values):.4f} "
                  f"(min: {np.min(values):.4f}, max: {np.max(values):.4f})")

        print(f"{'='*80}")

        # Plot distribution
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()

        for i, key in enumerate(['iou', 'dice', 'precision', 'recall', 'f1', 'sam_score']):
            values = [m[key] for m in all_metrics]
            axes[i].hist(values, bins=20, edgecolor='black', alpha=0.7)
            axes[i].axvline(np.mean(values), color='red', linestyle='--',
                          label=f'Mean: {np.mean(values):.3f}')
            axes[i].set_title(key.upper())
            axes[i].set_xlabel('Score')
            axes[i].set_ylabel('Count')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    return all_metrics

# Run evaluation
print("Starting SAM evaluation on SA-1B data...")

extracted_dir = '/content/drive/MyDrive/sam_dataset/minibatch'

# Make sure predictor is initialized
# predictor = ... (your SAM predictor)

metrics = evaluate_sam_on_folder(
    extracted_dir=extracted_dir,
    predictor=predictor,
    num_images=5,  # Evaluate 5 images
    min_gt_quality=0.90,  # Only use high-quality GT masks
    num_masks_per_image=3  # Evaluate 3 masks per image
)

Starting SAM evaluation on SA-1B data...


NameError: name 'predictor' is not defined

In [7]:
# ============================================================================
# QUICK TEST: Single Image
# ============================================================================

from pathlib import Path

extracted_dir = Path('/content/drive/MyDrive/sam_dataset/minibatch')

# Find first JSON file
json_files = list(extracted_dir.glob("**/*.json"))

if json_files:
    test_json = json_files[0]
    print(f"🔍 Testing with: {test_json}")

    # Load
    image, data = load_sa1b_image_and_annotations(test_json)

    print(f"📐 Image size: {data['image']['width']}x{data['image']['height']}")
    print(f"📊 Number of annotations: {len(data['annotations'])}")

    # Get first high-quality annotation
    high_quality = [
        ann for ann in data['annotations']
        if ann.get('predicted_iou', 0) > 0.9
    ]

    if high_quality:
        ann = high_quality[0]
        print(f"\n✅ Evaluating annotation with GT IoU: {ann['predicted_iou']:.3f}")

        pred_mask, gt_mask, metrics, bbox = evaluate_sam_on_annotation(
            image, ann, predictor
        )

        visualize_comparison(image, gt_mask, pred_mask, metrics, bbox,
                           title="Quick Test")

        print("\n📈 Metrics:")
        for key, value in metrics.items():
            print(f"  {key}: {value:.4f}")
    else:
        print("❌ No high-quality annotations found in this image")
else:
    print(f"❌ No JSON files found in {extracted_dir}")

🔍 Testing with: /content/drive/MyDrive/sam_dataset/extracted/sa_230132.json


NameError: name 'load_sa1b_image_and_annotations' is not defined